## IMPORTS

In [1]:
import genesis as gs
import numpy as np
import imageio
import torch
import os

# --- エラー回避用のモンキーパッチ ---
if not hasattr(torch._C._CudaDeviceProperties, 'shared_memory_per_multiprocessor'):
    torch._C._CudaDeviceProperties.shared_memory_per_multiprocessor = 102400
if not hasattr(torch._C._CudaDeviceProperties, 'shared_memory_per_block_optin'):
    torch._C._CudaDeviceProperties.shared_memory_per_block_optin = 99328

e:\matuo-PYZERO\.venv\Lib\site-packages\genesis\__init__.py:41: UserWarning: 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
  warn("'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/")


## 1. 環境構築

In [2]:
gs.init(backend=gs.cuda)
scene = gs.Scene(show_viewer=False)
plane = scene.add_entity(gs.morphs.Plane())

# MuJoCo MenagerieからUR5eを読み込み
robot = scene.add_entity(gs.morphs.MJCF(file="mujoco_menagerie/universal_robots_ur5e/ur5e.xml"))

# ターゲット（赤いブロック）を配置
target_cube = scene.add_entity(
    gs.morphs.Box(size=(0.04, 0.04, 0.04), pos=(0.6, 0.2, 0.025))
)

# 真上から見下ろす俯瞰カメラを設置
cam = scene.add_camera(
    res=(640, 480),
    pos=(0.5, 0.0, 1.0),
    lookat=(0.5, 0.0, 0.0),
    fov=60,
    GUI=False
)
scene.build()
print('Genesis環境を構築しました。')

[Genesis] [07:34:06] [WARNING] 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
[Genesis] [07:34:06] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [07:34:06] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [07:34:06] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [07:34:06] [INFO] Running on [NVIDIA GeForce RTX 3070 Ti] with backend gs.cuda. Device memory: 8.00 GB.
[Genesis] [07:34:06] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [07:34:06] [INFO] Scene <bba58e4> created.
[Genesis] [07:34:07] [INFO] Adding <gs.engine.entities.RigidEntity> plane_331904f. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [07:34:07] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properties.
[Genesis] [07:34:07] [WAR

e:\matuo-PYZERO\.venv\Lib\site-packages\quadrants\lang\ast\ast_transformers\function_def_transformer.py:781: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  return index == 0 and isinstance(stmt, ast.Expr) and isinstance(stmt.value, (ast.Constant, ast.Str))


[Genesis] [07:35:38] [INFO] Building visualizer...
Genesis環境を構築しました。


## --- AI・制御モジュールの定義 ---

In [3]:
class PointingAI:
    def __init__(self):
        # カメラの内部パラメータ
        self.H, self.W = 480, 640
        self.fov_y = np.deg2rad(60)
        self.fy = (self.H / 2) / np.tan(self.fov_y / 2)
        self.fx = self.fy 
        self.cx, self.cy = self.W / 2, self.H / 2
        
    def detect_target_uv(self, rgb_img):
        # 1. 画像からターゲットの (u, v) 座標を特定（色抽出による簡易CV）
        mask = (rgb_img[:,:,0] > 150) & (rgb_img[:,:,1] < 100) & (rgb_img[:,:,2] < 100)
        y_idx, x_idx = np.where(mask)
        if len(x_idx) > 0:
            u, v = int(np.mean(x_idx)), int(np.mean(y_idx))
            print(f"画像処理AI: ターゲットを画像上で検出 -> (u={u}, v={v})")
            return u, v
        
        print("画像処理AI: ターゲットが抽出できなかったため、推定位置を使用します。")
        return 430, 200 # フォールバック

    def estimate_3d_position(self, u, v, depth):
        # 2. (u, v) と Depth(深度) から実空間の3D座標を推定
        Z_c = depth if depth > 0 else 1.0
        x_c = (u - self.cx) * Z_c / self.fx
        y_c = (v - self.cy) * Z_c / self.fy
        
        # 俯瞰カメラからワールド座標系への変換 (簡易アフィン変換)
        X_world = 0.5 + y_c * 0.8 
        Y_world = 0.0 + x_c * 0.8 
        Z_world = 0.025 
        
        print(f"空間推定AI: 画像座標から実空間座標 (X={X_world:.3f}, Y={Y_world:.3f}) を推定。")
        return X_world, Y_world, Z_world

    def calculate_ik(self, target_x, target_y, target_z):
        # 3. 独自の簡易逆運動学 (IK)
        # 対象物の上空 (Z + 0.25) を指差すための関節角度を余弦定理で計算
        L1, L2, L3 = 0.16, 0.425, 0.392
        
        q0 = np.arctan2(target_y, target_x)
        r = np.sqrt(target_x**2 + target_y**2)
        z = (target_z + 0.25) - L1
        
        D = np.sqrt(r**2 + z**2)
        D = min(D, L2 + L3 - 0.01) 
        
        cos_q2 = (D**2 - L2**2 - L3**2) / (2 * L2 * L3)
        q2 = -np.arccos(np.clip(cos_q2, -1.0, 1.0)) 
        
        alpha = np.arctan2(z, r)
        cos_beta = (D**2 + L2**2 - L3**2) / (2 * D * L2)
        beta = np.arccos(np.clip(cos_beta, -1.0, 1.0))
        q1 = -(alpha + beta)
        
        # 指先を下に向ける姿勢
        q3 = -(np.pi/2 + q1 + q2)
        q4 = -np.pi/2
        q5 = 0.0
        
        return np.array([q0, q1, q2, q3, q4, q5])

# --- メイン実行ループ ---

In [4]:
def run_pointing_task():
    frames = []
    # 課題要件の「1分〜3分以内」を満たすため、1800ステップ(60秒)実行
    steps = 1800 
    
    # ロボットの初期姿勢 (少し肩を上げておく)
    init_q = np.zeros(robot.n_dofs)
    init_q[1] = -np.pi/2 
    robot.set_dofs_position(init_q)
    
    ai = PointingAI()
    
    # 1ステップ回してカメラ画像を更新
    scene.step()
    rgb, depth, _, _ = cam.render()
    if torch.is_tensor(rgb): rgb = rgb.detach().cpu().numpy()
    if torch.is_tensor(depth): depth = depth.detach().cpu().numpy()
        
    # 1. 画像からターゲットを検出
    u, v = ai.detect_target_uv(rgb)
    
    # 2. 3D空間座標の推定
    d_val = depth[v, u] if isinstance(depth, np.ndarray) and depth.ndim == 2 else 1.0
    X, Y, Z = ai.estimate_3d_position(u, v, d_val)
    
    # 3. 目標関節角度の計算 (独自IK)
    target_q = ai.calculate_ik(X, Y, Z)
    
    print('目標座標への滑らかな追従制御を開始します...')
    Kp = 5.0 # P制御のゲイン
    
    for step in range(steps):
        # 現在の関節角度を取得
        current_q = robot.get_dofs_position()
        if torch.is_tensor(current_q):
            current_q = current_q.detach().cpu().numpy()
            
        # 誤差に比例した速度を入力 (P制御によって滑らかに減速して止まる)
        error = target_q[:robot.n_dofs] - current_q
        velocity = Kp * error
        
        robot.control_dofs_velocity(velocity)
        scene.step()
        
        # 録画処理
        rgb_frame, _, _, _ = cam.render()
        if torch.is_tensor(rgb_frame):
            rgb_frame = rgb_frame.detach().cpu().numpy()
        frames.append(rgb_frame)
        
        if step > 0 and step % 300 == 0:
            print(f'Step {step}/{steps} completed. ({step//30}秒分録画完了)')
            
    print('動画のレンダリング中...')
    video_filename = 'pai_final_task_pointing.mp4'
    imageio.mimsave(video_filename, frames, fps=30)
    print(f'動画を保存しました: {video_filename}')
    print('課題への提出準備が完了しました！')

# 実行
run_pointing_task()

画像処理AI: ターゲットが抽出できなかったため、推定位置を使用します。
空間推定AI: 画像座標から実空間座標 (X=0.423, Y=0.212) を推定。
目標座標への滑らかな追従制御を開始します...
[Genesis] [07:35:43] [INFO] Running at 0.25 FPS.
[Genesis] [07:35:43] [INFO] Running at 2.24 FPS.
[Genesis] [07:35:43] [INFO] Running at 4.11 FPS.
[Genesis] [07:35:43] [INFO] Running at 6.09 FPS.
[Genesis] [07:35:43] [INFO] Running at 7.96 FPS.
[Genesis] [07:35:44] [INFO] Running at 9.53 FPS.
[Genesis] [07:35:44] [INFO] Running at 11.01 FPS.
[Genesis] [07:35:44] [INFO] Running at 12.44 FPS.
[Genesis] [07:35:44] [INFO] Running at 13.99 FPS.
[Genesis] [07:35:44] [INFO] Running at 15.57 FPS.
[Genesis] [07:35:44] [INFO] Running at 17.10 FPS.
[Genesis] [07:35:44] [INFO] Running at 19.04 FPS.
[Genesis] [07:35:44] [INFO] Running at 20.96 FPS.
[Genesis] [07:35:44] [INFO] Running at 23.12 FPS.
[Genesis] [07:35:44] [INFO] Running at 25.26 FPS.
[Genesis] [07:35:44] [INFO] Running at 27.20 FPS.
[Genesis] [07:35:44] [INFO] Running at 29.10 FPS.
[Genesis] [07:35:44] [INFO] Running at 30.94 FPS.
[

In [1]:
import genesis as gs
import numpy as np
import imageio
import torch
import os

# --- エラー回避用のモンキーパッチ ---
if not hasattr(torch._C._CudaDeviceProperties, 'shared_memory_per_multiprocessor'):
    torch._C._CudaDeviceProperties.shared_memory_per_multiprocessor = 102400
if not hasattr(torch._C._CudaDeviceProperties, 'shared_memory_per_block_optin'):
    torch._C._CudaDeviceProperties.shared_memory_per_block_optin = 99328

# 1. 環境構築
gs.init(backend=gs.cuda)
scene = gs.Scene(show_viewer=False)
plane = scene.add_entity(gs.morphs.Plane())

# MuJoCo MenagerieからUR5eを読み込み
robot = scene.add_entity(gs.morphs.MJCF(file="mujoco_menagerie/universal_robots_ur5e/ur5e.xml"))

# ターゲット（ブロック）を配置
target_cube = scene.add_entity(
    gs.morphs.Box(size=(0.04, 0.04, 0.04), pos=(0.6, 0.2, 0.025))
)

# 真上から見下ろす俯瞰カメラを設置
cam = scene.add_camera(
    res=(640, 480),
    pos=(0.5, 0.0, 1.0),
    lookat=(0.5, 0.0, 0.0),
    fov=60,
    GUI=False
)
scene.build()
print('Genesis環境を構築しました。')

# --- 制御用クラス ---
class PointingAI:
    def __init__(self):
        # 俯瞰カメラの視野を実空間にマッピングするための簡易係数
        self.x_scale = 1.0 / 480
        self.y_scale = 1.0 / 640

    def detect_target_uv(self, rgb_img):
        # 画像から白いブロックを抽出（背景の紺色を除外）
        mask = (rgb_img[:,:,0] > 150) & (rgb_img[:,:,1] > 150) & (rgb_img[:,:,2] > 150)
        y_idx, x_idx = np.where(mask)
        if len(x_idx) > 0:
            u, v = int(np.mean(x_idx)), int(np.mean(y_idx))
            print(f"画像処理AI: ターゲットを検出 -> (u={u}, v={v})")
            return u, v
        return 320, 240 # 見つからない場合は中央へ

    def estimate_3d_position(self, u, v):
        # カメラ画像 (u,v) から実空間の (X, Y) 座標を推定
        # ※ カメラ位置(0.5, 0.0) を中心とした簡易マッピング
        X_world = 0.5 - (v - 240) * self.x_scale * 0.8
        Y_world = 0.0 - (u - 320) * self.y_scale * 0.8
        Z_world = 0.25 # ブロックの少し上空
        
        print(f"空間推定AI: 実空間座標 (X={X_world:.3f}, Y={Y_world:.3f}, Z={Z_world:.3f}) を目標に設定")
        return np.array([X_world, Y_world, Z_world])

# --- メイン実行ループ ---
def run_pointing_task():
    frames = []
    steps = 1800 # 60秒
    
    # ロボットの初期姿勢をセット
    init_q = np.array([0.0, -1.57, 1.57, -1.57, -1.57, 0.0])
    robot.set_dofs_position(init_q)
    
    ai = PointingAI()
    
    # 1. 最初のフレームを取得してターゲット位置を推定
    scene.step()
    rgb, _, _, _ = cam.render()
    if torch.is_tensor(rgb): rgb = rgb.detach().cpu().numpy()
        
    u, v = ai.detect_target_uv(rgb)
    target_pos = ai.estimate_3d_position(u, v)
    
    # 手先を真下に向けるための回転（クォータニオン）
    target_quat = np.array([0.0, 1.0, 0.0, 0.0])
    
    # ★ Genesis内蔵のIKソルバーを使用して、目標座標から関節角度を計算
    end_effector = robot.get_link("wrist_3_link") # UR5eの先端リンク
    target_q = robot.inverse_kinematics(
        link=end_effector,
        pos=target_pos,
        quat=target_quat,
    )
    
    # IKの結果がGPUテンソルの場合はCPUのNumPy配列に変換
    if torch.is_tensor(target_q):
        target_q = target_q.detach().cpu().numpy().flatten()
    
    print('Genesis内蔵IKを用いた滑らかな追従制御を開始します...')
    Kp = 2.0 # ゲインを少し下げて、より滑らかに動かす
    
    for step in range(steps):
        current_q = robot.get_dofs_position()
        if torch.is_tensor(current_q):
            current_q = current_q.detach().cpu().numpy().flatten()
            
        # P制御で目標の関節角度へ近づける
        error = target_q[:robot.n_dofs] - current_q[:robot.n_dofs]
        velocity = Kp * error
        
        robot.control_dofs_velocity(velocity)
        scene.step()
        
        # 録画
        rgb_frame, _, _, _ = cam.render()
        if torch.is_tensor(rgb_frame):
            rgb_frame = rgb_frame.detach().cpu().numpy()
        frames.append(rgb_frame)
        
        if step > 0 and step % 300 == 0:
            print(f'Step {step}/{steps} completed. ({step//30}秒分録画完了)')
            
    print('動画のレンダリング中...')
    video_filename = 'pai_final_task_pointing.mp4'
    imageio.mimsave(video_filename, frames, fps=30)
    print(f'動画を保存しました: {video_filename}')

run_pointing_task()

e:\matuo-PYZERO\.venv\Lib\site-packages\genesis\__init__.py:41: UserWarning: 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
  warn("'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/")


[Genesis] [07:42:01] [WARNING] 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
[Genesis] [07:42:01] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [07:42:01] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [07:42:01] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [07:42:01] [INFO] Running on [NVIDIA GeForce RTX 3070 Ti] with backend gs.cuda. Device memory: 8.00 GB.
[Genesis] [07:42:01] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [07:42:01] [INFO] Scene <b61dc69> created.
[Genesis] [07:42:02] [INFO] Adding <gs.engine.entities.RigidEntity> plane_b1dd948. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [07:42:02] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properties.
[Genesis] [07:42:02] [WAR

e:\matuo-PYZERO\.venv\Lib\site-packages\quadrants\lang\ast\ast_transformers\function_def_transformer.py:781: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  return index == 0 and isinstance(stmt, ast.Expr) and isinstance(stmt.value, (ast.Constant, ast.Str))


[Genesis] [07:43:22] [INFO] Building visualizer...
Genesis環境を構築しました。
画像処理AI: ターゲットを検出 -> (u=65, v=298)
空間推定AI: 実空間座標 (X=0.403, Y=0.319, Z=0.250) を目標に設定
Genesis内蔵IKを用いた滑らかな追従制御を開始します...
[Genesis] [07:43:28] [INFO] Running at 0.20 FPS.
[Genesis] [07:43:28] [INFO] Running at 1.79 FPS.
[Genesis] [07:43:28] [INFO] Running at 3.60 FPS.
[Genesis] [07:43:28] [INFO] Running at 5.43 FPS.
[Genesis] [07:43:28] [INFO] Running at 8.02 FPS.
[Genesis] [07:43:28] [INFO] Running at 10.61 FPS.
[Genesis] [07:43:28] [INFO] Running at 13.08 FPS.
[Genesis] [07:43:28] [INFO] Running at 15.71 FPS.
[Genesis] [07:43:28] [INFO] Running at 18.06 FPS.
[Genesis] [07:43:28] [INFO] Running at 19.79 FPS.
[Genesis] [07:43:28] [INFO] Running at 21.76 FPS.
[Genesis] [07:43:28] [INFO] Running at 24.29 FPS.
[Genesis] [07:43:28] [INFO] Running at 26.30 FPS.
[Genesis] [07:43:28] [INFO] Running at 27.98 FPS.
[Genesis] [07:43:28] [INFO] Running at 29.91 FPS.
[Genesis] [07:43:28] [INFO] Running at 32.11 FPS.
[Genesis] [07:43:29]

3 state

In [1]:
import genesis as gs
import numpy as np
import imageio
import torch
import os

# --- エラー回避用のモンキーパッチ ---
if not hasattr(torch._C._CudaDeviceProperties, 'shared_memory_per_multiprocessor'):
    torch._C._CudaDeviceProperties.shared_memory_per_multiprocessor = 102400
if not hasattr(torch._C._CudaDeviceProperties, 'shared_memory_per_block_optin'):
    torch._C._CudaDeviceProperties.shared_memory_per_block_optin = 99328

# 1. 環境構築
gs.init(backend=gs.cuda)
scene = gs.Scene(show_viewer=False)
plane = scene.add_entity(gs.morphs.Plane())

# MuJoCo MenagerieからUR5eを読み込み
robot = scene.add_entity(gs.morphs.MJCF(file="mujoco_menagerie/universal_robots_ur5e/ur5e.xml"))

# 3つのターゲットブロックを異なる位置に配置
target_1 = scene.add_entity(gs.morphs.Box(size=(0.04, 0.04, 0.04), pos=(0.6, 0.2, 0.025)))
target_2 = scene.add_entity(gs.morphs.Box(size=(0.04, 0.04, 0.04), pos=(0.5, 0.0, 0.025)))
target_3 = scene.add_entity(gs.morphs.Box(size=(0.04, 0.04, 0.04), pos=(0.4, -0.2, 0.025)))

# 真上から見下ろす俯瞰カメラを設置
cam = scene.add_camera(
    res=(640, 480),
    pos=(0.5, 0.0, 1.0),
    lookat=(0.5, 0.0, 0.0),
    fov=60,
    GUI=False
)
scene.build()
print('Genesis環境を構築しました。')

# --- 制御用クラス ---
class PointingAI:
    def __init__(self):
        self.x_scale = 1.0 / 480
        self.y_scale = 1.0 / 640

    def estimate_3d_position(self, u, v):
        # カメラ画像上のピクセル座標 (u,v) から実空間の (X, Y, Z) 座標を推定する（逆投影）
        X_world = 0.5 - (v - 240) * self.x_scale * 0.8
        Y_world = 0.0 - (u - 320) * self.y_scale * 0.8
        Z_world = 0.25 # ブロックの上空（接触して壊さないための安全マージン）
        return np.array([X_world, Y_world, Z_world])

# --- メイン実行ループ ---
def run_sequential_pointing_task():
    frames = []
    steps = 1800 # 全体で60秒 (30fps)
    steps_per_target = 600 # 1つのターゲットにつき20秒待機・追従
    
    # 手動あるいは前段の物体検出AIが検知したとする、3つのターゲットの「カメラ画像上」のピクセル座標
    targets_uv = [
        (160, 180), # ターゲット1のピクセル位置
        (320, 240), # ターゲット2のピクセル位置
        (480, 300)  # ターゲット3のピクセル位置
    ]
    
    # ロボットの初期姿勢をセット
    init_q = np.array([0.0, -1.57, 1.57, -1.57, -1.57, 0.0])
    robot.set_dofs_position(init_q)
    
    ai = PointingAI()
    
    # 手先を真下に向ける姿勢（クォータニオン）
    target_quat = np.array([0.0, 1.0, 0.0, 0.0])
    end_effector = robot.get_link("wrist_3_link")
    
    print('シーケンス制御による連続ポインティングタスクを開始します...')
    Kp = 2.0 # P制御ゲイン
    
    current_target_idx = -1
    target_q = None
    
    for step in range(steps):
        # ステートマシンの切り替え判定 (20秒ごとに次のターゲットへ)
        target_idx = step // steps_per_target
        if target_idx != current_target_idx:
            current_target_idx = target_idx
            u, v = targets_uv[current_target_idx]
            
            # ピクセル座標から実空間の3D座標を推定
            target_pos = ai.estimate_3d_position(u, v)
            
            # Genesis内蔵IKで目標の関節角度を計算
            ik_result = robot.inverse_kinematics(
                link=end_effector,
                pos=target_pos,
                quat=target_quat,
            )
            if torch.is_tensor(ik_result):
                target_q = ik_result.detach().cpu().numpy().flatten()
            else:
                target_q = ik_result
                
            print(f"[{step}ステップ] 状態遷移: ターゲット{current_target_idx + 1} (ピクセル: {u}, {v}) を指差します。")

        # 現在の関節角度を取得
        current_q = robot.get_dofs_position()
        if torch.is_tensor(current_q):
            current_q = current_q.detach().cpu().numpy().flatten()
            
        # P制御による滑らかな追従
        error = target_q[:robot.n_dofs] - current_q[:robot.n_dofs]
        
        # 角度の差分を常に最短経路（-π 〜 π）に補正する
        error = np.arctan2(np.sin(error), np.cos(error))
        
        velocity = Kp * error
        
        robot.control_dofs_velocity(velocity)
        scene.step()
        
        # 録画
        rgb_frame, _, _, _ = cam.render()
        if torch.is_tensor(rgb_frame):
            rgb_frame = rgb_frame.detach().cpu().numpy()
        frames.append(rgb_frame)
        
    print('動画のレンダリング中...')
    video_filename = 'pai_final_task_sequential.mp4'
    imageio.mimsave(video_filename, frames, fps=30)
    print(f'動画を保存しました: {video_filename}')

run_sequential_pointing_task()

e:\matuo-PYZERO\.venv\Lib\site-packages\genesis\__init__.py:41: UserWarning: 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
  warn("'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/")


[Genesis] [07:55:18] [WARNING] 'torch<2.8.0' is not supported. Please upgrade pytorch manually: https://pytorch.org/get-started/locally/
[Genesis] [07:55:18] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [07:55:18] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [07:55:18] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [07:55:18] [INFO] Running on [NVIDIA GeForce RTX 3070 Ti] with backend gs.cuda. Device memory: 8.00 GB.
[Genesis] [07:55:18] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [07:55:18] [INFO] Scene <7ed86e7> created.
[Genesis] [07:55:19] [INFO] Adding <gs.engine.entities.RigidEntity> plane_6b406b2. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [07:55:19] [WARNING] Mesh is not watertight. Falling back to convex hull for estimating inertial properties.
[Genesis] [07:55:19] [WAR

e:\matuo-PYZERO\.venv\Lib\site-packages\quadrants\lang\ast\ast_transformers\function_def_transformer.py:781: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  return index == 0 and isinstance(stmt, ast.Expr) and isinstance(stmt.value, (ast.Constant, ast.Str))


[Genesis] [07:56:26] [INFO] Building visualizer...
Genesis環境を構築しました。
シーケンス制御による連続ポインティングタスクを開始します...
[0ステップ] 状態遷移: ターゲット1 (ピクセル: 160, 180) を指差します。
[Genesis] [07:56:32] [INFO] Running at 0.42 FPS.
[Genesis] [07:56:32] [INFO] Running at 2.04 FPS.
[Genesis] [07:56:32] [INFO] Running at 3.78 FPS.
[Genesis] [07:56:32] [INFO] Running at 6.39 FPS.
[Genesis] [07:56:32] [INFO] Running at 9.36 FPS.
[Genesis] [07:56:32] [INFO] Running at 12.02 FPS.
[Genesis] [07:56:32] [INFO] Running at 14.39 FPS.
[Genesis] [07:56:32] [INFO] Running at 16.91 FPS.
[Genesis] [07:56:32] [INFO] Running at 19.35 FPS.
[Genesis] [07:56:32] [INFO] Running at 21.82 FPS.
[Genesis] [07:56:32] [INFO] Running at 24.00 FPS.
[Genesis] [07:56:32] [INFO] Running at 26.30 FPS.
[Genesis] [07:56:32] [INFO] Running at 28.62 FPS.
[Genesis] [07:56:32] [INFO] Running at 30.75 FPS.
[Genesis] [07:56:32] [INFO] Running at 32.77 FPS.
[Genesis] [07:56:32] [INFO] Running at 34.61 FPS.
[Genesis] [07:56:33] [INFO] Running at 36.34 FPS.
[Genesis